# 3. New indicators

Calculates the derived indicators, adds them to the English file, and adds their
Arabic translation to the Arabic file.

```
COMPENDIUM-ARAB SOCIETY\<Chapter>_EN.xlsx          <- calculated rows appended
merged longfiles_AR\<Chapter>_AR.xlsx
        + the same rows, translated   ->   COMPENDIUM-ARAB SOCIETY\<Chapter>_AR.xlsx
```

| Indicator | Definition |
| --- | --- |
| Sex ratio, 2010-2025 (per 100 females) | male population / female population &times; 100 |
| Percentage of population by age group and by sex, 2024 | share of each sex in `<15`, `15-64`, `65+` |

## Only the new rows are translated

Earlier this notebook back-translated the entire English file into Arabic. That
meant pushing hundreds of thousands of rows through an inverted dictionary, and
inverting is lossy exactly where several Arabic spellings share one English
translation - 43 terms on the current data.

There is no need. The Arabic long file already exists, straight from the Arabic
questionnaires; it is the original, not a translation. All that is missing from
it are the rows this notebook invents. So only those are translated - a few
hundred rows, whose vocabulary is small and fully controlled.

Anything the dictionary does not know is collected for Claude Code to translate,
written back into `translation dict.xlsx`, and picked up on the next run.


In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config / paths


In [ ]:
"""
CELL: Configuration.
"""
DATA_COLLECTOR_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\DATA COLLECTOR")
TRANSLATION_DICT_PATH = DATA_COLLECTOR_PATH / "translation dict.xlsx"
COMPENDIUM_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY")


def long_file_folder(language):
    return COMPENDIUM_PATH / f"merged longfiles_{language}"


# Leave as None to do every chapter with a final English file, or restrict.
CHAPTERS = None

# The dictionary lives outside the repo, so git cannot track it. Every update
# also writes this CSV snapshot inside the repo: the .xlsx stays the working
# copy, the CSV is the versioned record - and CSV diffs cleanly where .xlsx
# does not.
DICTIONARY_SNAPSHOT_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY\codes\translation_dict_snapshot.csv")

# Columns whose values are numbers, never translated. Their headers still are.
NUMERIC_COLUMNS = {"Year", "Value"}


## The dictionary


In [ ]:
"""
CELL: Load the dictionary, and invert it for the handful of new rows.
"""


def load_dictionary():
    """The one dictionary, Arabic to English, plus the inverted maps.

    Inverting is only lossless where the mapping is one-to-one, and it is not
    always. That does not matter here the way it would for a whole-file
    back-translation: the only rows inverted are the ones this notebook creates,
    whose vocabulary is a couple of indicator titles, three age-group labels,
    the country names and the sexes. Collisions among those are reported anyway.
    """
    dict_df = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")

    column_map, value_map = {}, {}
    for arabic_column in dict_df["col_ar"].dropna().unique():
        rows = dict_df[dict_df["col_ar"] == arabic_column]
        column_map[arabic_column] = rows["col_en"].iloc[0]
        value_map[arabic_column] = {
            ar: en for ar, en in zip(rows["val_ar"], rows["val_en"]) if pd.notna(ar)
        }

    english_column = {}
    for arabic, english in column_map.items():
        english_column.setdefault(str(english).strip(), arabic)

    english_value, seen = {}, {}
    for _, row in dict_df.dropna(subset=["col_en", "val_en", "val_ar"]).iterrows():
        col = str(row["col_en"]).strip()
        val = str(row["val_en"]).strip()
        bucket = english_value.setdefault(col, {})
        if val in bucket:
            seen[(col, val)].append(row["val_ar"])
            continue
        bucket[val] = row["val_ar"]
        seen[(col, val)] = [row["val_ar"]]

    collisions = [(c, v, spellings) for (c, v), spellings in seen.items()
                  if len(spellings) > 1]
    return (column_map, value_map), (english_column, english_value), collisions


DICTIONARY_AR_TO_EN, ENGLISH_TO_ARABIC, COLLISIONS = load_dictionary()

arabic_columns, _ = DICTIONARY_AR_TO_EN
logger.info(f"Dictionary loaded: {len(arabic_columns)} column names")


## The calculations

Both run on the final English file, so a country that only submitted in English
is included.

Two indicators carry population by sex and age - `Population size by
nationality` (Total / Nationals / Non-nationals) and `Population size by area`
(Total / Urban / Rural). **They describe the same people**, so the code takes
only the **total** slice of whichever it uses, prefers the nationality one which
covers far more countries, and never adds the two together. Summing them, or
their parts, would count some people twice.

The denominator for the age shares is the sum of the three groups, not the
reported `Age Total`, so a country missing a band still adds to 100%.
`Age unknown` is excluded - it cannot be placed in a band.


In [ ]:
"""
CELL: The two calculations.
"""

AGE_GROUPS = {
    "<15 years": ["0-4 years", "5-9 years", "10-14 years"],
    "15-64 years": ["15-19 years", "20-24 years", "25-29 years", "30-34 years",
                    "35-39 years", "40-44 years", "45-49 years", "50-54 years",
                    "55-59 years", "60-64 years"],
    "65+ years": ["65-69 years", "70-74 years", "75+ years"],
}

SEX_RATIO_TITLE = "Sex ratio, 2010-2025 (per 100 females)"
AGE_SHARE_TITLE = "Percentage of population by age group and by sex, 2024"


def to_number(value):
    """Parse one Value cell into a float, or None if there is no number in it.

    These files store figures as text more often than as numbers, and not
    consistently: ' 701 956 ' uses spaces as thousand separators, some cells use
    commas, some carry a non-breaking space, and a few hold '-' for "no data".
    float() alone fails on all but the plainest of them.
    """
    if pd.isna(value):
        return None
    text = str(value).replace("\xa0", " ").replace(",", "").strip()
    text = re.sub(r"\s+", "", text)
    if text in ("", "-", "--", "..", "..."):
        return None
    try:
        return float(text)
    except ValueError:
        return None


def population_base(table):
    """Population counts as country / year / sex / age band, with no row counted
    twice. See the notes above for why the total slice is taken and why the two
    indicators are never added together."""
    frames = []
    for indicator, slice_column, slice_value in [
        ("Population size by nationality", "Nationality", "Nationality Total"),
        ("Population size by area", "Area", "Area Total"),
    ]:
        if slice_column not in table.columns:
            continue
        part = table[(table["Indicator"] == indicator)
                     & (table[slice_column] == slice_value)].copy()
        part["source_indicator"] = indicator
        frames.append(part)

    if not frames:
        return pd.DataFrame(columns=["Country", "Sex", "Age Group", "Year", "number"])

    combined = pd.concat(frames, ignore_index=True)
    combined["number"] = combined["Value"].map(to_number)
    combined = combined[combined["number"].notna()]

    preferred = "Population size by nationality"
    covered = set(combined.loc[combined["source_indicator"] == preferred, "Country"])
    keep = (combined["source_indicator"] == preferred) | (~combined["Country"].isin(covered))
    return combined[keep][["Country", "Sex", "Age Group", "Year", "number"]]


def calculate_sex_ratio(base, first_year=2010, last_year=2025):
    """One row per country and year: males per 100 females, all ages."""
    totals = base[(base["Age Group"] == "Age Total")
                  & (base["Sex"].isin(["Male", "Female"]))
                  & (base["Year"].astype(int).between(first_year, last_year))]

    wide = totals.pivot_table(index=["Country", "Year"], columns="Sex",
                              values="number", aggfunc="first")
    for needed in ("Male", "Female"):
        if needed not in wide.columns:
            logger.warning(f"Sex ratio: no '{needed}' rows - cannot calculate")
            return pd.DataFrame(columns=["Country", "Year", "Value"])
    wide = wide.dropna(subset=["Male", "Female"])
    wide = wide[wide["Female"] > 0]

    result = (wide["Male"] / wide["Female"] * 100).round(1).reset_index()
    result.columns = ["Country", "Year", "Value"]

    # Cross-check the reported all-ages total against the sum of the bands.
    bands = [b for group in AGE_GROUPS.values() for b in group]
    summed = (base[base["Age Group"].isin(bands)]
              .groupby(["Country", "Year", "Sex"])["number"].sum(min_count=1))
    reported = totals.set_index(["Country", "Year", "Sex"])["number"]
    shared = summed.index.intersection(reported.index)
    if len(shared):
        gap = (summed.loc[shared] - reported.loc[shared]).abs() / reported.loc[shared].abs()
        bad = gap[gap > 0.01]
        if len(bad):
            logger.warning(
                f"Sex ratio: {len(bad)} country/year/sex cell(s) where the reported "
                f"all-ages total and the sum of the age bands disagree by more than 1% "
                f"- the source figures contradict themselves:")
            for (country, year, sex), value in bad.sort_values(ascending=False).head(8).items():
                logger.warning(f"    {country} {year} {sex}: off by {value:.0%}")

    logger.info(f"Sex ratio: {len(result)} country/year value(s)")
    return result


def calculate_age_group_percentages(base, year=2024):
    """The percentage of each sex's population in each of the three age groups."""
    band_to_group = {band: group for group, bands in AGE_GROUPS.items() for band in bands}

    rows = base[(base["Year"].astype(int) == year)
                & (base["Sex"].isin(["Male", "Female"]))
                & (base["Age Group"].isin(band_to_group))].copy()
    if rows.empty:
        logger.warning(f"Age groups: no usable population rows for {year}")
        return pd.DataFrame(columns=["Country", "Sex", "Age Group", "Value"])

    rows["group"] = rows["Age Group"].map(band_to_group)
    by_group = rows.groupby(["Country", "Sex", "group"])["number"].sum(min_count=1)
    per_sex = by_group.groupby(["Country", "Sex"]).sum()

    result = (by_group / per_sex * 100).round(1).reset_index()
    result.columns = ["Country", "Sex", "Age Group", "Value"]
    result = result[result["Value"].notna()]

    check = result.groupby(["Country", "Sex"])["Value"].sum().round(0)
    off = check[(check - 100).abs() > 1]
    if len(off):
        logger.warning(f"Age groups: {len(off)} country/sex group(s) not summing to 100%:")
        for (country, sex), total in off.items():
            logger.warning(f"    {country} {sex}: {total}%")
    else:
        logger.info(f"Age groups: every country/sex adds to 100% ({len(check)} checked)")

    logger.info(f"Age groups {year}: {len(result)} value(s), "
                f"{result['Country'].nunique()} countries")
    return result


def calculated_rows(table, year_for_age_groups=2024):
    """The calculated indicator rows for one chapter's English table.

    Returns them as rows to append - it does not write anything, so the caller
    can put the same rows into the English file and, translated, into the
    Arabic one.
    """
    if "Indicator" not in table.columns:
        return pd.DataFrame()

    base = population_base(table)
    if base.empty:
        return pd.DataFrame()

    chapter_value = table["Chapter"].dropna().iloc[0] if "Chapter" in table.columns else None

    ratio = calculate_sex_ratio(base)
    ratio_rows = pd.DataFrame({
        "Indicator": SEX_RATIO_TITLE, "Country": ratio["Country"],
        "Year": ratio["Year"], "Value": ratio["Value"], "Chapter": chapter_value,
    })

    shares = calculate_age_group_percentages(base, year=year_for_age_groups)
    share_rows = pd.DataFrame({
        "Indicator": AGE_SHARE_TITLE, "Country": shares["Country"],
        "Sex": shares["Sex"], "Age Group": shares["Age Group"],
        "Year": year_for_age_groups, "Value": shares["Value"], "Chapter": chapter_value,
    })

    return pd.concat([ratio_rows, share_rows], ignore_index=True)


## Translating just the new rows


In [ ]:
"""
CELL: translate_new_rows() - the calculated rows, into Arabic.
"""


def looks_arabic(text):
    """True if the text contains at least one Arabic letter."""
    return any("\u0600" <= c <= "\u06ff" for c in str(text))


def translate_new_rows(rows):
    """Translate the calculated rows into Arabic, and report what could not be.

    Returns (arabic_rows, gaps). Values are replaced first and the column renamed
    second, because the value lookup is keyed by the column's ORIGINAL English
    name. Year and Value keep their numbers; only their headers change.
    """
    english_column, english_value = ENGLISH_TO_ARABIC
    out = rows.copy()
    gaps = []

    for column in list(out.columns):
        if column not in NUMERIC_COLUMNS:
            mapping = english_value.get(column, {})
            present = out[column].dropna().astype(str).str.strip().unique()
            for value in present:
                if value in mapping:
                    continue
                if looks_arabic(value) or not re.search(r"[A-Za-z]", value):
                    continue          # already Arabic, or a number/code
                gaps.append({"col_en": column,
                             "col_ar": english_column.get(column, column),
                             "val_en": value, "val_ar": None,
                             "rows": int((out[column].astype(str).str.strip() == value).sum())})
            if mapping:
                out[column] = out[column].replace(mapping)
        if column in english_column:
            out = out.rename(columns={column: english_column[column]})

    return out, gaps


## Filling the gaps

The calculated indicators invent labels no questionnaire contains, so the
dictionary can only learn them from here.

### Letting Claude Code close them for you

Ask once:

> **"run the pipeline and fill any dictionary gaps"**

Claude Code runs the notebook, translates whatever the dictionary did not know,
writes it back into `translation dict.xlsx` (taking a backup first), re-runs to
confirm, and reports what it added. The protocol is recorded in `CLAUDE.md`.

By hand it is the same two calls: `export_gaps(REPORTS)` -> fill in `val_ar` ->
`update_dictionary(filled)`.


In [ ]:
"""
CELL: export_gaps() and update_dictionary().
"""


def export_gaps(reports, file_name="new_indicator_labels.xlsx"):
    """Write the labels with no Arabic to a file shaped like the dictionary."""
    rows = [g for r in reports for g in r["gaps"]]
    if not rows:
        logger.info("Nothing to fill in - the dictionary knew every label.")
        return pd.DataFrame()
    gaps = (pd.DataFrame(rows).drop_duplicates(subset=["col_en", "val_en"])
            .sort_values(["col_en", "val_en"]).reset_index(drop=True))
    path = COMPENDIUM_PATH / file_name
    gaps.to_excel(path, index=False, engine="openpyxl")
    logger.info(f"Saved {path.name}: {len(gaps)} label(s). Fill in val_ar, "
                f"then call update_dictionary().")
    return gaps


def update_dictionary(filled, backup=True):
    """Append reviewed translations to translation dict.xlsx.

    Rows missing either side are skipped, and an (Arabic column, Arabic value)
    pair already present is left alone - so running this twice changes nothing
    the second time. A timestamped backup is written first, because this edits
    the project's source of truth.
    """
    if not isinstance(filled, pd.DataFrame):
        filled = pd.read_excel(filled, engine="openpyxl")

    needed = ["col_ar", "val_ar", "col_en", "val_en"]
    missing = [c for c in needed if c not in filled.columns]
    if missing:
        raise ValueError(f"missing column(s) {missing}; expected {needed}")

    new_rows = filled[needed].dropna()
    new_rows = new_rows[(new_rows["val_ar"].astype(str).str.strip() != "")
                        & (new_rows["val_en"].astype(str).str.strip() != "")]
    if new_rows.empty:
        logger.warning("No completed rows to add - is val_ar filled in?")
        return None

    dictionary = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")
    already = set(zip(dictionary["col_ar"], dictionary["val_ar"]))
    to_add = new_rows[~new_rows.apply(
        lambda r: (r["col_ar"], r["val_ar"]) in already, axis=1)]
    if to_add.empty:
        logger.info("Every row is already in the dictionary - nothing to add.")
        return dictionary

    if backup:
        stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        backup_path = TRANSLATION_DICT_PATH.with_name(
            f"{TRANSLATION_DICT_PATH.stem} backup {stamp}.xlsx")
        dictionary.to_excel(backup_path, index=False, engine="openpyxl")
        logger.info(f"Backed up the dictionary to {backup_path.name}")

    updated = pd.concat([dictionary, to_add.reindex(columns=dictionary.columns)],
                        ignore_index=True)
    updated.to_excel(TRANSLATION_DICT_PATH, index=False, engine="openpyxl")
    updated.to_csv(DICTIONARY_SNAPSHOT_PATH, index=False, encoding="utf-8-sig")
    logger.info(f"Added {len(to_add):,} row(s) to {TRANSLATION_DICT_PATH.name} "
                f"({len(dictionary):,} -> {len(updated):,}). Re-run this notebook.")
    return updated


## Building one chapter


## `build_chapter()`


In [ ]:
"""
CELL: build_chapter() - calculate, append to English, translate, append to Arabic.
"""


def chapters_to_process():
    if CHAPTERS:
        return list(CHAPTERS)
    found = sorted(p.name[: -len("_EN.xlsx")] for p in COMPENDIUM_PATH.glob("*_EN.xlsx"))
    logger.info(f"Chapters with a final English file: {found}")
    if not found:
        logger.warning("No <Chapter>_EN.xlsx found - run notebook 2 first.")
    return found


def build_chapter(chapter):
    """Adds the calculated indicators to the English file, and their translation
    to the Arabic one. Returns a dict describing what happened."""
    english_path = COMPENDIUM_PATH / f"{chapter}_EN.xlsx"
    arabic_source = long_file_folder("AR") / f"{chapter}_AR.xlsx"
    arabic_path = COMPENDIUM_PATH / f"{chapter}_AR.xlsx"

    report = {"chapter": chapter, "new_rows": 0, "english_rows": 0,
              "arabic_rows": 0, "gaps": []}

    if not english_path.exists():
        logger.info(f"  {chapter}: no {english_path.name}, skipping")
        return report

    english = pd.read_excel(english_path, engine="openpyxl")

    # Drop anything a previous run added, so this is repeatable.
    if "Indicator" in english.columns:
        added = english["Indicator"].isin([SEX_RATIO_TITLE, AGE_SHARE_TITLE])
        if added.any():
            logger.info(f"  {chapter}: removing {added.sum():,} row(s) from a previous run")
            english = english[~added]

    new_rows = calculated_rows(english)
    if new_rows.empty:
        logger.info(f"  {chapter}: no population rows, nothing to calculate")
        return report
    report["new_rows"] = len(new_rows)

    combined_english = pd.concat([english, new_rows], ignore_index=True)
    combined_english.to_excel(english_path, index=False, engine="openpyxl")
    report["english_rows"] = len(combined_english)
    logger.info(f"  {chapter}: {len(new_rows):,} calculated row(s) -> {english_path.name} "
                f"({len(combined_english):,} rows)")

    # ---------------------------------------------------------- the Arabic side
    if not arabic_source.exists():
        logger.warning(f"  {chapter}: no {arabic_source.name} - run notebook 1 first, "
                       f"so the Arabic file cannot be built")
        return report

    arabic_new, gaps = translate_new_rows(new_rows)
    report["gaps"] = gaps

    arabic = pd.read_excel(arabic_source, engine="openpyxl")
    indicator_ar = ENGLISH_TO_ARABIC[0].get("Indicator", "Indicator")
    titles_ar = [ENGLISH_TO_ARABIC[1].get("Indicator", {}).get(t, t)
                 for t in (SEX_RATIO_TITLE, AGE_SHARE_TITLE)]
    if indicator_ar in arabic.columns:
        previous = arabic[indicator_ar].isin(titles_ar)
        if previous.any():
            arabic = arabic[~previous]

    combined_arabic = pd.concat([arabic, arabic_new], ignore_index=True)
    combined_arabic.to_excel(arabic_path, index=False, engine="openpyxl")
    report["arabic_rows"] = len(combined_arabic)
    logger.info(f"  {chapter}: {arabic_source.name} + {len(arabic_new):,} translated "
                f"row(s) -> {arabic_path.name} ({len(combined_arabic):,} rows)")
    return report

## Run


In [ ]:
"""
CELL: Main run.
"""
print("Adding the calculated indicators")
print(f"  English: {COMPENDIUM_PATH}\\<Chapter>_EN.xlsx")
print(f"  Arabic : merged longfiles_AR + the translated rows -> <Chapter>_AR.xlsx\n")

if COLLISIONS:
    logger.info(f"{len(COLLISIONS)} English term(s) have several Arabic spellings; "
                f"the first is used where one is needed.")

REPORTS = []
chapters = chapters_to_process()
for i, chapter in enumerate(chapters, start=1):
    bar = "#" * i + "-" * (len(chapters) - i)
    print(f"[{bar}] chapter {i}/{len(chapters)}: {chapter}")
    REPORTS.append(build_chapter(chapter))

print("\n" + "=" * 74)
print(f"{'Chapter':<12}{'calculated':>12}{'English total':>15}{'Arabic total':>15}")
for r in REPORTS:
    print(f"{r['chapter']:<12}{r['new_rows']:>12,}{r['english_rows']:>15,}{r['arabic_rows']:>15,}")

gap_count = len({(g["col_en"], g["val_en"]) for r in REPORTS for g in r["gaps"]})
print()
if gap_count == 0:
    print("Every calculated label translated - the dictionary knew them all.")
else:
    print(f"{gap_count} label(s) in the new rows have no Arabic in the dictionary:")
    shown = set()
    for r in REPORTS:
        for g in r["gaps"]:
            key = (g["col_en"], g["val_en"])
            if key in shown:
                continue
            shown.add(key)
            print(f"   [{g['col_en']}] {g['val_en']}")
    print("\nRun export_gaps(REPORTS), have Claude Code fill in val_ar, then")
    print("update_dictionary() - or just ask it to fill the dictionary gaps.")


## Gaps


In [ ]:
"""
CELL: Write the gap file for the run above.
"""
NEW_LABELS = export_gaps(REPORTS)
NEW_LABELS.head(20)
